# Data Description
- Background: Unit 1.7 補充資料驗證：道路路寬標註。提供機關：臺北市都市發展局；官方來源：https://data.taipei/dataset/detail?id=068be2c7-eea1-4303-91f8-fbf50c9b2aec。原始來源對應與官方最新資源bytes完全相同為不同命題；除歷史界內包已逐byte雜湊對應外，未重新下載認證其他最新資源。
- date received: 學生於本單元提供；實際下載／收到日期未提供。首次本機檢視2026-11-26，不將今日當下載日。
- Path to data file: 官方來源及版本如上；精確本機容器如下，解壓至data/temp僅為衍生暫存。
  - `data/raw/路寬檔_提供.zip`；1,062,574 bytes；SHA-256 `a92d7b7225509489181606158d1d9cbe3b1a4c83b120ee12d577460109852d70`。
- Unit of observation: 計畫／地籍參考路寬標註尺寸線，非道路中心線。
- Sample period: ZIP內資料修改2023-12-28，XML製作2021；非2024／2025完整年度實測；發布／更新：資源2023-12-28；metadata2026-06-17。ZIP時間、檔名時間、觀測／設置時間分開，不由檔名猜全年存量。
- Known issues: 官方稱依都市計畫參酌地籍標註；線長與標註寬度相近，不能當道路縱向長度作權重或實測車行空間。缺失、重複、坐標及CRS由下方實讀輸出；沒有完整性證據的0僅為本檔未觀測。授權：政府資料開放授權條款第1版；依官方頁，來源與版本須保留。
- Definition for each variable: 下方逐欄型態、缺失、觀測域與定義表。已知官方欄意與本次診斷欄分開；未獲欄位碼表支持者明記未知，不依名稱猜設施單位。
- Data path: 上列每個獨立原始容器；實際讀取成員列於Summary Stats。
- Role: raw。
- Verification status: 以本Notebook乾淨kernel實際輸出與執行metadata為準；不認證已清理。
- SHA-256: 上列每檔固定綁定，載入前逐項重核。
- Next processing notebook: 本單元僅診斷，Chapter 2尚未建立；[驗證一](../code/taipei_supplementary_accident_boundary_validation.ipynb)、[驗證二](../code/taipei_transportation_infrastructure_validation.ipynb)。

Author: 詹明翰、謝子詮、曾翊涵；Created At / Last Modified At: 2026-11-26。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。共同作者與職務不代表已完成個別工作。本輪由Codex協助執行，成員驗收與審查尚待完成。


In [1]:
from pathlib import Path
import sys, hashlib, json, zipfile
import pandas as pd
from IPython.display import display, Markdown
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate/'AGENTS.md').is_file() and (candidate/'templates').is_dir():
        PROJECT_ROOT=candidate;break
else:raise FileNotFoundError('請從本研究工作包內執行')
def fingerprint(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_config.json')=='7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7'
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_support.py')=='1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080'
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
from unit17_validation_support import Context, show, safe_text
pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',160)
ctx=Context(PROJECT_ROOT,'7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7')
print('Python:',sys.version,'Config:','7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7','Helper:','1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080')


Python: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)] Config: 7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7 Helper: 1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080


## Change for Your Own Data

使用已核對精確路徑及固定SHA-256；不選第一個glob或最新檔。CP950非法bytes只作可逆診斷；大檔分批讀取、OSM僅研究區2km bbox，保留全國母體數。

In [2]:
FAMILY='road_width'
INPUTS=ctx.family_items(FAMILY)
for item in INPUTS:ctx.file(item['path'])
frames=ctx.load_family(FAMILY)

## Summary

每個容器、實際成員與載入範圍分列；沒有將暫存解壓檔當原始下載資料。

In [3]:
show('固定輸入身分',pd.DataFrame([{k:v for k,v in r.items() if k not in ['members','encoding_checks']} for r in INPUTS]))
show('讀取範圍與CRS／編碼',pd.DataFrame(ctx.summaries[FAMILY]))

**固定輸入身分**

,family,path,bytes,sha256,status
0,road_width,data/raw/路寬檔_提供.zip,1062574,a92d7b7225509489181606158d1d9cbe3b1a4c83b120ee12d577460109852d70,present


**讀取範圍與CRS／編碼**

,path,member,total_features,loaded_rows,crs,types,bounds,invalid,null_geometry,empty
0,data/raw/路寬檔_提供.zip,roadsize2.shp,22676,22676,EPSG:3826,{'LineString': 22676},"[296588.423088, 2763381.375987, 313023.928664, 2782648.787537]",0,0,0


### Sample

各實際讀取表前十列；全國事故為臺北篩選後第一當事人，不代表全國前十列。OSM為bbox空間子集。

In [4]:
for source,frame in frames.items():
    show(source,frame.drop(columns='geometry',errors='ignore').head(10),block=10)

**data/raw/路寬檔_提供.zip::roadsize2.shp**

,road_width,angle,_source_row,_source_path,_source_member
fid,,,,,
0,8M,-11.24,1,data/raw/路寬檔_提供.zip,roadsize2.shp
1,6M,26.56,2,data/raw/路寬檔_提供.zip,roadsize2.shp
2,6M,-37.63,3,data/raw/路寬檔_提供.zip,roadsize2.shp
3,6M,30.82,4,data/raw/路寬檔_提供.zip,roadsize2.shp
4,8M,38.65,5,data/raw/路寬檔_提供.zip,roadsize2.shp
5,6M,-84.70,6,data/raw/路寬檔_提供.zip,roadsize2.shp
6,8M,-11.24,7,data/raw/路寬檔_提供.zip,roadsize2.shp
7,6M,-3.68,8,data/raw/路寬檔_提供.zip,roadsize2.shp
8,6M,91.28,9,data/raw/路寬檔_提供.zip,roadsize2.shp


### Summary Stats

ZIP逐成員CRC、配套、原始WKT、全部欄型態／缺失／定義、重複與數值／類別／時間診斷。保留原值及bytes。

In [5]:
_ = ctx.raw_report(FAMILY,crc=True)

,source_metadata_and_limits
title,道路路寬標註
provider,臺北市都市發展局
url,https://data.taipei/dataset/detail?id=068be2c7-eea1-4303-91f8-fbf50c9b2aec
publication,資源2023-12-28；metadata2026-06-17
period,ZIP內資料修改2023-12-28，XML製作2021；非2024／2025完整年度實測
limitation,官方稱依都市計畫參酌地籍標註；線長與標註寬度相近，不能當道路縱向長度作權重或實測車行空間
license,政府資料開放授權條款第1版；依官方頁，來源與版本須保留


,family,path,bytes,sha256,status
0,road_width,data/raw/路寬檔_提供.zip,1062574,a92d7b7225509489181606158d1d9cbe3b1a4c83b120ee12d577460109852d70,present


**ZIP原始成員清單；非解壓暫存資料來源**

,member,bytes,compressed,crc32,zip_datetime
0,roadsize2.cpg,5,5,0e813c50,"[2023, 3, 13, 10, 10, 56]"
1,roadsize2.dbf,476294,82632,a949b803,"[2023, 12, 28, 17, 11, 40]"
2,roadsize2.prj,399,256,90c78104,"[2021, 3, 29, 16, 56, 6]"
3,roadsize2.sbn,225964,119367,0a36a320,"[2021, 3, 29, 16, 56, 12]"
4,roadsize2.sbx,12500,5987,e21cb836,"[2021, 3, 29, 16, 56, 12]"
5,roadsize2.shp,1995588,799994,fcf93d71,"[2023, 12, 28, 3, 0, 26]"
6,roadsize2.shp.xml,6131,2088,394cd1ff,"[2021, 3, 29, 17, 8, 12]"
7,roadsize2.shx,181508,51399,fa73f5f0,"[2023, 12, 28, 3, 0, 26]"


ZIP 全成員CRC通過: data/raw/路寬檔_提供.zip


**實際讀取範圍／編碼／CRS／全國或地區計數**

... (8 outputs left out)

,field,dtype,missing,distinct,definition
0,road_width,str,0,134,都市計畫／地籍路寬標註原字串，非實測車行寬度
1,angle,str,0,10487,原角度字串，單位及精確用途未獲欄位規格認證
2,_source_row,int64,0,22676,本次來源追溯列號；不是官方穩定事件或設施ID
3,_source_path,str,0,1,本次僅記憶體追溯／診斷旗標，原始檔未新增欄
4,_source_member,str,0,1,本次僅記憶體追溯／診斷旗標，原始檔未新增欄


**數值摘要（不把類別碼大小當實質尺度）**

,index,count,mean,std,min,25%,50%,75%,max
0,_source_row,22676.0,11338.5,6546.141688,1.0,5669.75,11338.5,17007.25,22676.0


原始屬性完整列重複（不含診斷追溯欄） 6634


本文件只確認資料身分及可讀性；語義／日期不足者限制用途，不強行納入正式Dataset B3。人類驗收仍待完成。

In [6]:
for item in INPUTS:ctx.file(item['path'])
print('原始容器hash重核通過；正式分析資料輸出0。')

原始容器hash重核通過；正式分析資料輸出0。
